In [56]:
from astropy.io import fits
import matplotlib.pyplot as plt
import numpy as np
from scipy.optimize import curve_fit
from astropy.modeling import models
from scipy.signal import convolve
from scipy.optimize import minimize
from astropy.convolution import Gaussian1DKernel
import scipy.integrate as integrate
from scipy.optimize import differential_evolution
import pandas as pd
import os
from scipy.integrate import simpson

In [57]:
hdul = fits.open("C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/Mrk_544/FORS2.2021-08-21T05_32_16.048/Reduced/Mrk-544_MAPPED_FLUX_SCI_LSS_U1.fits")
hdul.info()
image = hdul[0].data
image_error = hdul[1].data

Filename: C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/Mrk_544/FORS2.2021-08-21T05_32_16.048/Reduced/Mrk-544_MAPPED_FLUX_SCI_LSS_U1.fits
No.    Name      Ver    Type      Cards   Dimensions   Format
  0  PRIMARY       1 PrimaryHDU     365   (2094, 964)   float32   
  1  IMAGE.ERR     1 ImageHDU        46   (2094, 964)   float32   


In [58]:
image_cut = image[55:155, 0:1800]
image_error_cut = image_error[55:155, 0:1800]

In [59]:
wv = np.arange(4300.74, 4300.74+1800*1.48, 1.48)
J, N = image_cut.data.shape
#x = np.arange(0, N, 1)
y = np.arange(0, J, 1)
arcsec_y = np.arange(-58*0.256, 42*0.256, 0.256)

z = 0.0238
l_HA = 6563 * (1+z)
l_HB = 4861 * (1+z) 
l_NII = 6583 * (1+z)
l_SII = 6717 * (1+z)
l_OIII = 5007 * (1+z)

In [60]:
def calculate_bn(n):
    """ Calcola b_n per il profilo Sérsic """
    return 2*n - 1/3 + 4/(405*n) + 46/(25515*n**2)

def sersic_1d(x, params):
    """ Profilo Sérsic 1D centrato in x_0 """
    I_e, r_e, n, x_0 = params
    r = np.abs(x - x_0)
    b_n = calculate_bn(n)
    return I_e * np.exp(-b_n * ((r / r_e)**(1 / n) - 1))
    
def sersic_1d_convolved(x, params):
    """ Sérsic convoluto con PSF gaussiana """
    profile = sersic_1d(x, params)
    psf = Gaussian1DKernel(stddev = sigma, mode='center')
    conv_profile = convolve(profile, psf, mode='same')
    return conv_profile

def model_convolved(x, params1, params2, params3, params4):
    x_hr = np.linspace(min(x), max(x), 4000)
    profile = (
        sersic_1d(x_hr, params1) +
        sersic_1d(x_hr, params2) +
        sersic_1d(x_hr, params3) +
        sersic_1d(x_hr, params4)
    )
    kernel = Gaussian1DKernel(stddev=(sigma*len(x_hr))/len(x), x_size=len(x_hr), mode='center')   
    conv_profile = convolve(profile, kernel, mode='same')
    conv_profile_out = np.interp(x, x_hr, conv_profile)
    return conv_profile_out

In [61]:
seeing = 3.174
sigma = seeing / 2.35 
x_axes = np.arange(0, 100, 1)

In [62]:
def single_integral(x, params, a, b):
    # x --> array di x 
    # y --> dati sulla y (sersic_1d_conv(x))
    # params --> parametri del profilo 
    # a e b --> estremi di integrazione
    func = sersic_1d_convolved(x, params)
    mask = (x >= a)*(x <= b)
    integral = simpson(y = func[mask], x = x[mask])
    return integral 

def total_integral(x, params1, params2, params3, params4, a, b):
    func = model_convolved(x, params1, params2, params3, params4)
    mask = (x >= a)*(x <= b)
    integral = simpson(y = func[mask], x = x[mask])
    return integral  

def contamination(x, params1, params2, params3, params4, a, b): 
    blue_area = single_integral(x, params1, a, b) 
    green_area = single_integral(x, params2, a, b)
    purple_area = single_integral(x, params3, a, b)
    orange_area = single_integral(x, params4, a, b)
    sum_area = blue_area + green_area +purple_area + orange_area
    total_area = total_integral(x, params1, params2, params3, params4, a, b)
    print(sum_area)
    print(total_area)
    return [(blue_area*100)/sum_area, (green_area*100)/sum_area, (purple_area*100)/sum_area, (orange_area*100)/sum_area]     

## Halpha

In [63]:
mask_HA = (wv > l_HA-8)*(wv < l_HA+8)
image_HA = image_cut[:, mask_HA]
image_error_HA = image_error_cut[:, mask_HA]
radial_profile_HA = np.sum(image_HA, axis = 1)
radial_profile_error_HA = np.sqrt(np.sum(image_error_HA**2, axis = 1))
fit = pd.read_csv("C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/Mrk_544/FORS2.2021-08-21T05_32_16.048/Reduced/Fit_code/Halpha_fit.csv", index_col = 0)

In [64]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit['Component 1'].iloc[3] -sigma, 
                                                                fit['Component 1'].iloc[3] + sigma)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit['Component 2'].iloc[3] -sigma, 
                                                                fit['Component 2'].iloc[3] + sigma)
data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit['Component 3'].iloc[3] -sigma, 
                                                                fit['Component 3'].iloc[3] + sigma)
data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit['Component 4'].iloc[3] -sigma, 
                                                                fit['Component 4'].iloc[3] + sigma)
data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1), np.array(data4).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(4)],
                  index=['Blue', 'Green', 'Purple', 'Orange'])
print(df)

df_path = os.path.join('Areas_Halpha_1s.csv')
df.to_csv(df_path, float_format='%.3f')

22.351550355585776
22.013004474783166
9.649401031242252
9.35642330776773
3.818623196591535
4.990547726296186
1.7330328850382632
1.7492334339265971
           Peak 1     Peak 2     Peak 3     Peak 4
Blue    95.410068  40.167152   2.870771   3.325759
Green    3.881093  57.144859   0.400300   0.082258
Purple   0.491786   1.948873  93.074378  19.472625
Orange   0.217053   0.739116   3.654551  77.119357


In [65]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit['Component 1'].iloc[3] -2*sigma, 
                                                                fit['Component 1'].iloc[3] + 2*sigma)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit['Component 2'].iloc[3] -2*sigma, 
                                                                fit['Component 2'].iloc[3] + 2*sigma)
data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit['Component 3'].iloc[3] -2*sigma, 
                                                                fit['Component 3'].iloc[3] + 2*sigma)
data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit['Component 4'].iloc[3] -2*sigma, 
                                                                fit['Component 4'].iloc[3] + 2*sigma)
data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1), np.array(data4).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(4)],
                  index=['Blue', 'Green', 'Purple', 'Orange'])
print(df)

df_path = os.path.join('Areas_Halpha_2s.csv')
df.to_csv(df_path, float_format='%.3f')

40.45743348081264
39.92348820270982
18.663130889944107
18.202264475081012
15.93585395607583
19.864872390883406
3.3433287871830193
3.369503435974817
           Peak 1     Peak 2     Peak 3     Peak 4
Blue    94.634544  43.720439   3.493798   3.466254
Green    4.579230  53.483386   0.511886   0.087664
Purple   0.545851   2.029842  91.566292  20.446530
Orange   0.240375   0.766334   4.428024  75.999552


In [66]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit['Component 1'].iloc[3] -3*sigma, 
                                                                fit['Component 1'].iloc[3] + 3*sigma)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit['Component 2'].iloc[3] -3*sigma, 
                                                                fit['Component 2'].iloc[3] + 3*sigma)
data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit['Component 3'].iloc[3] -3*sigma, 
                                                                fit['Component 3'].iloc[3] + 3*sigma)
data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit['Component 4'].iloc[3] -3*sigma, 
                                                                fit['Component 4'].iloc[3] + 3*sigma)
data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1), np.array(data4).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(4)],
                  index=['Blue', 'Green', 'Purple', 'Orange'])
print(df)

df_path = os.path.join('Areas_Halpha_3s.csv')
df.to_csv(df_path, float_format='%.3f')

59.45762936495089
58.81512172058598
36.81429211391679
36.33515377112942
19.355633956090777
23.45357500415028
5.373670964025878
5.40517273269897
           Peak 1     Peak 2     Peak 3     Peak 4
Blue    91.157239  55.613796   4.096745   3.649122
Green    7.859835  41.481457   0.632803   0.091520
Purple   0.685783   2.119395  90.103039  21.568086
Orange   0.297143   0.785351   5.167413  74.691273


## HBeta

In [67]:
mask_HB = (wv > l_HB-8)*(wv < l_HB+8)
image_HB = image_cut[:, mask_HB] 
image_error_HB = image_error_cut[:, mask_HB]
radial_profile_HB = np.sum(image_HB, axis = 1)
radial_profile_error_HB = np.sqrt(np.sum(image_error_HB**2, axis = 1))
fit = pd.read_csv("C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/Mrk_544/FORS2.2021-08-21T05_32_16.048/Reduced/Fit_code/Hbeta_fit.csv", index_col = 0)

In [68]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit['Component 1'].iloc[3] -sigma, 
                                                                fit['Component 1'].iloc[3] + sigma)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit['Component 2'].iloc[3] -sigma, 
                                                                fit['Component 2'].iloc[3] + sigma)
data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit['Component 3'].iloc[3] -sigma, 
                                                                fit['Component 3'].iloc[3] + sigma)
data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit['Component 4'].iloc[3] -sigma, 
                                                                fit['Component 4'].iloc[3] + sigma)
data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1), np.array(data4).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(4)],
                  index=['Blue', 'Green', 'Purple', 'Orange'])
print(df)

df_path = os.path.join('Areas_Hbeta_1s.csv')
df.to_csv(df_path, float_format='%.3f')

6.558599574291017
6.237525795922405
3.043973381874731
3.0592013179039292
1.8736135799034952
2.1498631345816324
0.9362249615501856
0.947449530732748
           Peak 1     Peak 2     Peak 3     Peak 4
Blue    89.233949  47.942105   1.133817   0.829178
Green    8.482128  45.046270   1.645480   1.085304
Purple   1.996916   6.173688  94.417445  33.674211
Orange   0.287007   0.837937   2.803258  64.411306


In [69]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit['Component 1'].iloc[3] -2*sigma, 
                                                                fit['Component 1'].iloc[3] + 2*sigma)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit['Component 2'].iloc[3] -2*sigma, 
                                                                fit['Component 2'].iloc[3] + 2*sigma)
data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit['Component 3'].iloc[3] -2*sigma, 
                                                                fit['Component 3'].iloc[3] + 2*sigma)
data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit['Component 4'].iloc[3] -2*sigma, 
                                                                fit['Component 4'].iloc[3] + 2*sigma)
data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1), np.array(data4).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(4)],
                  index=['Blue', 'Green', 'Purple', 'Orange'])
print(df)

df_path = os.path.join('Areas_Hbeta_2s.csv')
df.to_csv(df_path, float_format='%.3f')

12.03278931532971
11.521999179787981
7.169486948095087
7.193187951901854
8.030981705847898
8.948527753019595
1.80691484741774
1.8250707767808567
           Peak 1     Peak 2     Peak 3     Peak 4
Blue    88.019744  48.430373   1.352537   0.866654
Green    9.482800  43.768508   1.964219   1.135798
Purple   2.183922   6.879055  93.372511  35.196556
Orange   0.313533   0.922064   3.310732  62.800992


In [70]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit['Component 1'].iloc[3] -3*sigma, 
                                                                fit['Component 1'].iloc[3] + 3*sigma)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit['Component 2'].iloc[3] -3*sigma, 
                                                                fit['Component 2'].iloc[3] + 3*sigma)
data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit['Component 3'].iloc[3] -3*sigma, 
                                                                fit['Component 3'].iloc[3] + 3*sigma)
data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit['Component 4'].iloc[3] -3*sigma, 
                                                                fit['Component 4'].iloc[3] + 3*sigma)
data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1), np.array(data4).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(4)],
                  index=['Blue', 'Green', 'Purple', 'Orange'])
print(df)

df_path = os.path.join('Areas_Hbeta_3s.csv')
df.to_csv(df_path, float_format='%.3f')


17.625992930886987
17.029175817175133
10.265691541741642
10.258550386662458
9.962603701111894
10.915165631628302
2.9002300705547963
2.9219867629992033
           Peak 1     Peak 2     Peak 3     Peak 4
Blue    86.159289  52.892340   1.564159   0.908210
Green   10.928507  39.408151   2.273094   1.187669
Purple   2.546344   6.793399  92.374937  37.266853
Orange   0.365860   0.906110   3.787809  60.637269


## NII

In [71]:
mask_NII = (wv > l_NII-8)*(wv < l_NII+8)
image_NII = image_cut[:, mask_NII]
image_error_NII = image_error_cut[:, mask_NII]
radial_profile_NII = np.sum(image_NII, axis = 1)
radial_profile_error_NII = np.sqrt(np.sum(image_error_NII**2, axis = 1))
fit = pd.read_csv("C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/Mrk_544/FORS2.2021-08-21T05_32_16.048/Reduced/Fit_code/NII_fit.csv", index_col = 0)

In [72]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit['Component 1'].iloc[3] -sigma, 
                                                                fit['Component 1'].iloc[3] + sigma)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit['Component 2'].iloc[3] -sigma, 
                                                                fit['Component 2'].iloc[3] + sigma)
data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit['Component 3'].iloc[3] -sigma, 
                                                                fit['Component 3'].iloc[3] + sigma)
data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit['Component 4'].iloc[3] -sigma, 
                                                                fit['Component 4'].iloc[3] + sigma)
data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1), np.array(data4).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(4)],
                  index=['Blue', 'Green', 'Purple', 'Orange'])
print(df)

df_path = os.path.join('Areas_NII_1s.csv')
df.to_csv(df_path, float_format='%.3f')

5.476625381206358
5.413105448850109
3.234528807130359
3.095542643986532
2.610686152743473
3.2438860588236755
1.1879098234935104
1.195383177627332
           Peak 1     Peak 2     Peak 3     Peak 4
Blue    87.696950  33.456895   1.558699   2.016093
Green    9.127770  58.250659   1.734141   1.999560
Purple   2.341273   6.306202  92.641302  28.490676
Orange   0.834007   1.986244   4.065857  67.493672


In [73]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit['Component 1'].iloc[3] -2*sigma, 
                                                                fit['Component 1'].iloc[3] + 2*sigma)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit['Component 2'].iloc[3] -2*sigma, 
                                                                fit['Component 2'].iloc[3] + 2*sigma)
data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit['Component 3'].iloc[3] -2*sigma, 
                                                                fit['Component 3'].iloc[3] + 2*sigma)
data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit['Component 4'].iloc[3] -2*sigma, 
                                                                fit['Component 4'].iloc[3] + 2*sigma)
data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1), np.array(data4).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(4)],
                  index=['Blue', 'Green', 'Purple', 'Orange'])
print(df)

df_path = os.path.join('Areas_NII_2s.csv')
df.to_csv(df_path, float_format='%.3f')

10.102499352631034
10.001527856434377
6.242793102219117
6.023316033692906
11.06929026904894
13.189574197073142
2.3161092188648356
2.328265395764714
           Peak 1     Peak 2     Peak 3     Peak 4
Blue    86.357665  36.110273   1.862416   2.077448
Green   10.189062  55.255230   2.078368   2.062025
Purple   2.547403   6.571915  91.223040  29.521125
Orange   0.905870   2.062582   4.836176  66.339401


In [74]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit['Component 1'].iloc[3] -3*sigma, 
                                                                fit['Component 1'].iloc[3] + 3*sigma)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit['Component 2'].iloc[3] -3*sigma, 
                                                                fit['Component 2'].iloc[3] + 3*sigma)
data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit['Component 3'].iloc[3] -3*sigma, 
                                                                fit['Component 3'].iloc[3] + 3*sigma)
data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit['Component 4'].iloc[3] -3*sigma, 
                                                                fit['Component 4'].iloc[3] + 3*sigma)
data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1), np.array(data4).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(4)],
                  index=['Blue', 'Green', 'Purple', 'Orange'])
print(df)

df_path = os.path.join('Areas_NII_3s.csv')
df.to_csv(df_path, float_format='%.3f')

15.477062141744145
15.343259755334131
11.999978555098517
11.75603222000599
13.614574759637764
15.824511709347817
3.78926252136229
3.804158082299432
           Peak 1     Peak 2     Peak 3     Peak 4
Blue    81.583212  44.928771   2.150675   2.152366
Green   14.303165  45.909471   2.408162   2.132629
Purple   3.044567   6.997481  89.883811  30.644821
Orange   1.069056   2.164276   5.557352  65.070184


## SII

In [75]:
mask_SII = (wv > l_SII-8)*(wv < l_SII+8)
x_axes = np.arange(0, 100, 1)
image_SII = image_cut[:, mask_SII] 
image_error_SII = image_error_cut[:, mask_SII]
radial_profile_SII = np.sum(image_SII, axis = 1)
radial_profile_error_SII = np.sqrt(np.sum(image_error_SII**2, axis = 1))
fit = pd.read_csv("C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/Mrk_544/FORS2.2021-08-21T05_32_16.048/Reduced/Fit_code/SII_fit.csv", index_col = 0)

In [76]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit['Component 1'].iloc[3] -sigma, 
                                                                fit['Component 1'].iloc[3] + sigma)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit['Component 2'].iloc[3] -sigma, 
                                                                fit['Component 2'].iloc[3] + sigma)
data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit['Component 3'].iloc[3] -sigma, 
                                                                fit['Component 3'].iloc[3] + sigma)
data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit['Component 4'].iloc[3] -sigma, 
                                                                fit['Component 4'].iloc[3] + sigma)
data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1), np.array(data4).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(4)],
                  index=['Blue', 'Green', 'Purple', 'Orange'])
print(df)

df_path = os.path.join('Areas_SII_1s.csv')
df.to_csv(df_path, float_format='%.3f')

2.7262777589558667
2.7050908720433933
1.7899828364968193
1.7640718789521312
1.7616465730782245
2.071714622709383
0.8414354060012124
0.8482979205260859
           Peak 1     Peak 2     Peak 3     Peak 4
Blue    81.992390  37.663696   2.089171   2.965561
Green   13.571990  51.920649   2.430080   2.852319
Purple   2.904548   7.315040  91.042430  26.716110
Orange   1.531071   3.100614   4.438319  67.466009


In [77]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit['Component 1'].iloc[3] -2*sigma, 
                                                                fit['Component 1'].iloc[3] + 2*sigma)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit['Component 2'].iloc[3] -2*sigma, 
                                                                fit['Component 2'].iloc[3] + 2*sigma)
data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit['Component 3'].iloc[3] -2*sigma, 
                                                                fit['Component 3'].iloc[3] + 2*sigma)
data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit['Component 4'].iloc[3] -2*sigma, 
                                                                fit['Component 4'].iloc[3] + 2*sigma)
data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1), np.array(data4).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(4)],
                  index=['Blue', 'Green', 'Purple', 'Orange'])
print(df)

df_path = os.path.join('Areas_SII_2s.csv')
df.to_csv(df_path, float_format='%.3f')

5.1269682717373275
5.093113985558015
3.5155709852694974
3.4749104995838946
7.534096730696776
8.568852803770644
1.6380197254494488
1.6491554355217954
           Peak 1     Peak 2     Peak 3     Peak 4
Blue    80.590363  39.456239   2.467562   3.057788
Green   14.677760  49.885306   2.878238   2.944902
Purple   3.101411   7.495975  89.425705  27.746481
Orange   1.630466   3.162479   5.228495  66.250830


In [78]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit['Component 1'].iloc[3] -3*sigma, 
                                                                fit['Component 1'].iloc[3] + 3*sigma)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit['Component 2'].iloc[3] -3*sigma, 
                                                                fit['Component 2'].iloc[3] + 3*sigma)
data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit['Component 3'].iloc[3] -3*sigma, 
                                                                fit['Component 3'].iloc[3] + 3*sigma)
data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit['Component 4'].iloc[3] -3*sigma, 
                                                                fit['Component 4'].iloc[3] + 3*sigma)
data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1), np.array(data4).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(4)],
                  index=['Blue', 'Green', 'Purple', 'Orange'])
print(df)

df_path = os.path.join('Areas_SII_3s.csv')
df.to_csv(df_path, float_format='%.3f')

8.126825739980042
8.084287176978568
6.904122598578761
6.860891357057754
9.327713072696898
10.404315020919473
2.6724662315371663
2.6859635225460727
           Peak 1     Peak 2     Peak 3     Peak 4
Blue    76.050947  45.422647   2.821704   3.184345
Green   18.501724  43.504541   3.301364   3.053703
Purple   3.598733   7.831352  87.914775  28.844748
Orange   1.848596   3.241460   5.962156  64.917204


## OIII

In [79]:
mask_OIII = (wv > l_OIII-8)*(wv < l_OIII+8)
image_OIII = image_cut[:, mask_OIII] 
image_error_OIII = image_error_cut[:, mask_OIII]
radial_profile_OIII = np.sum(image_OIII, axis = 1)
radial_profile_error_OIII = np.sqrt(np.sum(image_error_OIII**2, axis = 1))
fit = pd.read_csv("C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/Mrk_544/FORS2.2021-08-21T05_32_16.048/Reduced/Fit_code/OIII_fit.csv", index_col = 0)

In [80]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit['Component 1'].iloc[3] -sigma, 
                                                                fit['Component 1'].iloc[3] + sigma)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit['Component 2'].iloc[3] -sigma, 
                                                                fit['Component 2'].iloc[3] + sigma)
data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit['Component 3'].iloc[3] -sigma, 
                                                                fit['Component 3'].iloc[3] + sigma)
data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit['Component 4'].iloc[3] -sigma, 
                                                                fit['Component 4'].iloc[3] + sigma)
data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1), np.array(data4).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(4)],
                  index=['Blue', 'Green', 'Purple', 'Orange'])
print(df)

df_path = os.path.join('Areas_OIII_1s.csv')
df.to_csv(df_path, float_format='%.3f')

8.106929903831624
7.997879640681596
2.629100095120288
2.631730033340012
2.068721488391658
2.3876993858953135
1.0939534722735191
1.1010482184811365
           Peak 1     Peak 2     Peak 3        Peak 4
Blue    93.153008  60.922011   3.326088  4.146580e+00
Green    4.347246  27.640229   0.015709  1.581204e-08
Purple   1.807965   8.612150  91.726740  3.309134e+01
Orange   0.691781   2.825610   4.931463  6.276208e+01


In [81]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit['Component 1'].iloc[3] -2*sigma, 
                                                                fit['Component 1'].iloc[3] + 2*sigma)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit['Component 2'].iloc[3] -2*sigma, 
                                                                fit['Component 2'].iloc[3] + 2*sigma)
data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit['Component 3'].iloc[3] -2*sigma, 
                                                                fit['Component 3'].iloc[3] + 2*sigma)
data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit['Component 4'].iloc[3] -2*sigma, 
                                                                fit['Component 4'].iloc[3] + 2*sigma)
data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1), np.array(data4).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(4)],
                  index=['Blue', 'Green', 'Purple', 'Orange'])
print(df)

df_path = os.path.join('Areas_OIII_2s.csv')
df.to_csv(df_path, float_format='%.3f')

14.75198438996488
14.581120214721725
5.371919467724543
5.377818934018373
8.943612514275651
10.007590625179436
2.138498406726404
2.150048691004144
           Peak 1     Peak 2     Peak 3        Peak 4
Blue    92.428916  62.191940   3.890992  4.258366e+00
Green    4.816392  26.566171   0.030694  3.984134e-08
Purple   1.993379   8.471825  90.334689  3.415023e+01
Orange   0.761313   2.770063   5.743625  6.159140e+01


In [82]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit['Component 1'].iloc[3] -3*sigma, 
                                                                fit['Component 1'].iloc[3] + 3*sigma)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit['Component 2'].iloc[3] -3*sigma, 
                                                                fit['Component 2'].iloc[3] + 3*sigma)
data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit['Component 3'].iloc[3] -3*sigma, 
                                                                fit['Component 3'].iloc[3] + 3*sigma)
data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], 
                                                                fit['Component 4'], fit['Component 4'].iloc[3] -3*sigma, 
                                                                fit['Component 4'].iloc[3] + 3*sigma)
data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1), np.array(data4).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(4)],
                  index=['Blue', 'Green', 'Purple', 'Orange'])
print(df)

df_path = os.path.join('Areas_OIII_3s.csv')
df.to_csv(df_path, float_format='%.3f')

21.67974825601102
21.49137214423169
11.956383182303197
11.985612984129428
11.168674963115286
12.275304408823402
3.5157814185438374
3.5299120800530166
           Peak 1     Peak 2     Peak 3        Peak 4
Blue    90.127194  67.531712   4.417202  4.401238e+00
Green    6.466549  22.195692   0.053493  1.085670e-07
Purple   2.475607   7.768010  89.040117  3.530455e+01
Orange   0.930650   2.504586   6.489188  6.029421e+01
